### Load Library

In [2]:
import pandas as pd
import os
import zipfile
import requests

### Download Data

In [11]:
base_dir = "data/archive"
os.makedirs(base_dir, exist_ok=True)

zip_path = os.path.join(base_dir, "OSD-944_metadata_OSD-944-ISA.zip")
file1 = os.path.join(base_dir, "a_OSD-944_metagenomic-sequencing_short-read-low-biomass-dna-sequencing_Illumina.txt")
file2 = os.path.join(base_dir, "s_OSD-944.txt")

download_url = "http://nasa-osdr.s3.amazonaws.com/OSD-944/version-1/metadata/OSD-944_metadata_OSD-944-ISA.zip"

# Step 1: Check if required files exist
if os.path.exists(file1) and os.path.exists(file2):
    print("Metadata files already exist.")

else:
    # Step 2: Check if ZIP exists
    if os.path.exists(zip_path):
        print("ZIP file found.")
    else:
        print("ZIP not found. Downloading...")

        response = requests.get(download_url, stream=True)
        response.raise_for_status()

        with open(zip_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        print("Download complete.")

    # Step 3: Unzip
    print("Unzipping...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(base_dir)

    print("Done.")

print(f"Metadata file 1 exists: {os.path.exists(file1)}")
print(f"Metadata file 2 exists: {os.path.exists(file2)}")

Metadata files already exist.
Metadata file 1 exists: True
Metadata file 2 exists: True


### Load Data

In [12]:
input_file = "data/archive/s_OSD-944.txt"
output_file = "data/clean_metadata.tsv"

In [13]:
print("Loading raw metadata...")
df = pd.read_csv(input_file, sep='\t')
print(df.shape)
# print(df.columns)
# df

Loading raw metadata...
(38, 27)


### Copy Data

In [15]:
cols_to_keep = [
    'Sample Name',
    'Characteristics[Biospecimen Type]',
    'Factor Value[Sample Location]',
    'Factor Value[Collection Date]',
    'Parameter Value[Prep Number]'
]

clean_df = df[cols_to_keep].copy()

clean_df.columns = [
    'Sample_ID', 
    'Sample_Type', 
    'Location', 
    'Collection_Date', 
    'Prep_Number'
]

print(clean_df.shape)
print(clean_df["Sample_Type"].value_counts())

# clean_df.head()
clean_df

(38, 5)
Sample_Type
Environmental    24
Control          14
Name: count, dtype: int64


,Sample_ID,Sample_Type,Location,Collection_Date,Prep_Number
0,D1S1,Environmental,Floor by gowning room,27-May-2025,"1,2,3"
1,D1S2,Environmental,Floor by airlock,27-May-2025,"1,2,3"
2,D1S3,Environmental,Floor by main door,27-May-2025,"1,2,3"
3,D1S4,Environmental,Floor by cabinets,27-May-2025,"1,2,3"
4,D1S5,Environmental,Field Control,27-May-2025,"1,2,3"
5,D1S6,Environmental,Floor mid room,27-May-2025,"1,2,3"
6,D1S7,Environmental,Field control,27-May-2025,"1,2,3"
7,D1S8,Environmental,Floor by spacecraft,27-May-2025,"1,2,3"
8,D1S9,Environmental,Tabletop,27-May-2025,"1,2,3"
9,D1S10,Environmental,Floor by emergency exit,27-May-2025,"1,2,3"


### Clean & Save (Control + Environment)

In [16]:
# Replace the specific Sample_ID string
clean_df['Sample_ID'] = clean_df['Sample_ID'].replace('Zymo_MMC_Log_11ng', 'Zymo_LD_MMC')

In [17]:
clean_df = clean_df[
    clean_df['Prep_Number']
    .astype(str)
    .str.split(',')
    .apply(lambda prep_list: '3' in [item.strip() for item in prep_list])
].reset_index(drop=True)

clean_df.to_csv(output_file, sep='\t', index=False)
print(f"Cleaned metadata saved to {output_file}")
# clean_df

Cleaned metadata saved to data/clean_metadata.tsv


In [18]:
clean_df

,Sample_ID,Sample_Type,Location,Collection_Date,Prep_Number
0,D1S1,Environmental,Floor by gowning room,27-May-2025,"1,2,3"
1,D1S2,Environmental,Floor by airlock,27-May-2025,"1,2,3"
2,D1S3,Environmental,Floor by main door,27-May-2025,"1,2,3"
3,D1S4,Environmental,Floor by cabinets,27-May-2025,"1,2,3"
4,D1S5,Environmental,Field Control,27-May-2025,"1,2,3"
5,D1S6,Environmental,Floor mid room,27-May-2025,"1,2,3"
6,D1S7,Environmental,Field control,27-May-2025,"1,2,3"
7,D1S8,Environmental,Floor by spacecraft,27-May-2025,"1,2,3"
8,D1S9,Environmental,Tabletop,27-May-2025,"1,2,3"
9,D1S10,Environmental,Floor by emergency exit,27-May-2025,"1,2,3"
